# D335 — Sqoop Practical: Amazon RDS MySQL ↔ Amazon EMR

Perform one complete round trip:

```text
RDS MySQL customers table -> Sqoop import -> HDFS
HDFS customer_scores file -> Sqoop export -> RDS MySQL customer_scores table
```

**Assumptions:** an `emr-6.15.0` cluster with Sqoop is running; an RDS MySQL instance is running; both use the same default VPC and can communicate on TCP 3306. RDS/EMR creation and network configuration are outside this notebook.

Replace every placeholder before running. The example password is for a disposable demonstration database only.

## 1. Values used in this lab

| Value | Example |
|---|---|
| EMR primary DNS | `ec2-...compute.amazonaws.com` |
| SSH key | `/path/to/dataeng.pem` |
| RDS endpoint | `mydb.abc123.ap-south-1.rds.amazonaws.com` |
| RDS port | `3306` |
| MySQL master username | `admin` |
| MySQL password | `DataEng@12345` |
| Database | `sqoop_lab` |

Do not include `https://` in the RDS endpoint. The endpoint is a hostname, not a web URL.

## 2. Create source and destination tables in RDS MySQL

Connect with MySQL Workbench or any MySQL client as `admin`, then run:

```sql
CREATE DATABASE IF NOT EXISTS sqoop_lab;
USE sqoop_lab;

DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    city VARCHAR(60),
    signup_date DATE NOT NULL
);

INSERT INTO customers VALUES
(101, 'Asha',  'Chennai',   '2026-08-01'),
(102, 'Ravi',  'Bengaluru', '2026-08-03'),
(103, 'Meera', 'Pune',       '2026-08-05'),
(104, 'Kabir', 'Hyderabad',  '2026-08-07');

DROP TABLE IF EXISTS customer_scores;
CREATE TABLE customer_scores (
    customer_id INT PRIMARY KEY,
    score INT NOT NULL,
    segment VARCHAR(20) NOT NULL
);

SELECT * FROM customers ORDER BY customer_id;
SELECT * FROM customer_scores;
```

The export destination must exist before Sqoop runs. It is empty now so inserts cannot collide with existing primary keys.

## 3. SSH to the EMR primary node

Run from the **local terminal**:

```bash
chmod 400 /path/to/dataeng.pem
ssh -i /path/to/dataeng.pem hadoop@EMR_PRIMARY_PUBLIC_DNS
```

All remaining shell commands run on the EMR primary node unless a section says otherwise.

## 4. Configure lab variables and password file

Replace `RDS_ENDPOINT` with the actual RDS endpoint. Run as `hadoop` on EMR:

```bash
export RDS_ENDPOINT='replace-me.abcdefgh.ap-south-1.rds.amazonaws.com'
export DB_PORT='3306'
export DB_NAME='sqoop_lab'
export DB_USER='admin'
export JDBC_URL="jdbc:mariadb://${RDS_ENDPOINT}:${DB_PORT}/${DB_NAME}"
export PASSWORD_FILE='/home/hadoop/.d335-rds-password'

printf '%s' 'DataEng@12345' > "${PASSWORD_FILE}"
chmod 400 "${PASSWORD_FILE}"
```

The quotes prevent shell interpretation of `@` and other characters. `--password-file` keeps the password out of the Sqoop command line, though the file still contains a secret and must be removed during cleanup. Avoid placing credentials in notebooks used beyond this disposable lab.

## 5. Verify EMR, Sqoop, driver, DNS, and TCP access

```bash
cat /emr/instance-controller/lib/info/extraInstanceData.json | grep releaseLabel
sqoop version
ls /usr/lib/sqoop/lib/*mariadb*
getent hosts "${RDS_ENDPOINT}
timeout 5 bash -c "</dev/tcp/${RDS_ENDPOINT}/${DB_PORT}" && echo 'RDS port reachable'
```

Expected release is `emr-6.15.0`; Sqoop is 1.4.7. AWS installs a MariaDB JDBC driver with Sqoop. MariaDB Connector/J speaks to the MySQL server, therefore use the MariaDB URL and driver class without downloading a JAR.

If DNS or TCP fails, stop here: Sqoop cannot repair connectivity.

## 6. Test JDBC metadata access

```bash
sqoop list-tables \
  --connect "${JDBC_URL}" \
  --driver org.mariadb.jdbc.Driver \
  --username "${DB_USER}" \
  --password-file "file://${PASSWORD_FILE}"
```

Expected names include `customers` and `customer_scores`. This proves the endpoint, credentials, database name, driver, and basic permissions are usable before launching MapReduce.

## 7. Import one MySQL table into HDFS

Remove only the previous D335 target if it exists, then import:

```bash
hdfs dfs -rm -r -f /user/hadoop/d335/customers

sqoop import \
  --connect "${JDBC_URL}" \
  --driver org.mariadb.jdbc.Driver \
  --username "${DB_USER}" \
  --password-file "file://${PASSWORD_FILE}" \
  --table customers \
  --columns 'customer_id,customer_name,city,signup_date' \
  --target-dir /user/hadoop/d335/customers \
  --fields-terminated-by ',' \
  --null-string '\\N' \
  --null-non-string '\\N' \
  --num-mappers 1
```

`--num-mappers 1` is intentional for four rows. With multiple mappers, specify/test a suitable indexed split column such as `customer_id` and consider the load placed on RDS.

## 8. Validate the import

```bash
hdfs dfs -ls /user/hadoop/d335/customers
hdfs dfs -cat /user/hadoop/d335/customers/part-m-*
hdfs dfs -cat /user/hadoop/d335/customers/part-m-* | wc -l
```

Expected row count: `4`. The files are in HDFS on EMR core-node storage. They are not automatically durable after cluster termination.

A simple count validates volume, not correctness. Real pipelines also check keys, nulls, ranges, duplicates, source/target counts at a controlled point in time, and representative records.

## 9. Create an HDFS file for export

Create three rows in the same order as the MySQL destination columns:

```bash
printf '101,92,gold\n102,81,silver\n103,88,gold\n' > /tmp/d335_customer_scores.csv

hdfs dfs -mkdir -p /user/hadoop/d335/customer_scores
hdfs dfs -put -f /tmp/d335_customer_scores.csv /user/hadoop/d335/customer_scores/part-00000
hdfs dfs -cat /user/hadoop/d335/customer_scores/part-00000
```

The order is `customer_id,score,segment`, matching the `--columns` list in the next step.

## 10. Export one HDFS dataset to MySQL

```bash
sqoop export \
  --connect "${JDBC_URL}" \
  --driver org.mariadb.jdbc.Driver \
  --username "${DB_USER}" \
  --password-file "file://${PASSWORD_FILE}" \
  --table customer_scores \
  --columns 'customer_id,score,segment' \
  --export-dir /user/hadoop/d335/customer_scores \
  --input-fields-terminated-by ',' \
  --input-null-string '\\N' \
  --input-null-non-string '\\N' \
  --num-mappers 1
```

Sqoop inserts rows by default. If this job partially succeeds and is rerun, primary-key duplicates may fail. Production flows often export to a staging table and then use database-side SQL to validate and merge idempotently.

## 11. Validate the export in RDS

Run in MySQL Workbench or a MySQL client:

```sql
USE sqoop_lab;
SELECT customer_id, score, segment
FROM customer_scores
ORDER BY customer_id;
```

Expected result:

| customer_id | score | segment |
|---:|---:|---|
| 101 | 92 | gold |
| 102 | 81 | silver |
| 103 | 88 | gold |

The successful round trip demonstrates both directions; it does not imply that Sqoop continuously synchronizes the systems.

## 12. Common failures

| Symptom | Likely cause | Check |
|---|---|---|
| Connection refused/timed out | Endpoint/port connectivity | Endpoint, DNS, TCP 3306 |
| Access denied | Username/password or MySQL grants | Login and database privileges |
| No suitable driver/class not found | JDBC URL/driver mismatch | `/usr/lib/sqoop/lib`, `--driver` |
| Target directory already exists | Previous import output | Use a new path or deliberately remove lab path |
| Cannot split table | Multiple mappers but no suitable key | Use `--split-by`, or `-m 1` for a small lab |
| Export parse/type error | Delimiter, order, nulls, or types differ | HDFS rows and destination DDL |
| Duplicate-key export error | Rows were already committed | Inspect target; use staging/idempotent design |
| One mapper is much slower | Skewed split ranges | Key distribution and mapper logs |

YARN application logs are the first place to inspect task failures: `yarn application -list -appStates ALL`, followed by `yarn logs -applicationId APPLICATION_ID`.

## 13. Optional production concepts (do not run here)

- Parallel imports: `--num-mappers 4 --split-by customer_id` after checking distribution and RDS capacity.
- Filtered import: `--where "signup_date >= '2026-08-01'"`.
- Query import: requires `$CONDITIONS` in the SQL when Sqoop parallelizes it.
- Incremental append: `--incremental append --check-column customer_id --last-value ...`; externalize and manage the state.
- Incremental last-modified: handles updates differently and still needs careful late-data/delete semantics.
- Hive import or Parquet/Avro output where supported and tested.
- Export update mode with `--update-key` and `--update-mode`, or preferably a controlled staging/merge workflow.

Sqoop is retired; consider whether a maintained JDBC, Glue, DMS, or CDC solution better fits any new pipeline.

## 14. Cleanup

On EMR, remove only the D335 files and secret:

```bash
hdfs dfs -rm -r -f /user/hadoop/d335
rm -f /tmp/d335_customer_scores.csv
rm -f /home/hadoop/.d335-rds-password
unset RDS_ENDPOINT DB_PORT DB_NAME DB_USER JDBC_URL PASSWORD_FILE
```

In MySQL, if the entire lab database can be discarded:

```sql
DROP DATABASE IF EXISTS sqoop_lab;
```

The database drop is destructive and cannot be assumed recoverable. Run it only for the disposable `sqoop_lab` database. EMR and RDS termination are separate and outside this notebook.

## 15. Verification checklist and references

Verify the following outcomes:

- `sqoop import`: RDS MySQL table to HDFS;
- `sqoop export`: HDFS records to a pre-created RDS MySQL table;
- why the RDS endpoint, JDBC driver, delimiter, split column, and mapper count matter;
- how partial exports affect retries; and
- why Sqoop 1.4.7 should be treated as legacy technology.

Official references: [Sqoop on EMR](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-sqoop.html), [EMR Sqoop considerations](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-sqoop-considerations.html), [Sqoop 1.4.7 User Guide](https://sqoop.apache.org/docs/1.4.7/SqoopUserGuide.html), and [Apache Attic status](https://attic.apache.org/projects/sqoop.html).